In [ ]:
# Install required libraries!pip install -q xgboost tensorflow shap

In [ ]:
# Import librariesimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport warningswarnings.filterwarnings('ignore')from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,                              confusion_matrix, classification_report, roc_curve, auc)import xgboost as xgbimport tensorflow as tffrom tensorflow.keras.models import Sequentialfrom tensorflow.keras.layers import LSTM, Dense, Dropoutsns.set_style('whitegrid')np.random.seed(42)tf.random.set_seed(42)

In [ ]:
# Generate synthetic vehicle sensor datasetn_samples = 6000engine_temperature = np.random.normal(90, 12, n_samples)rpm                = np.random.normal(2500, 500, n_samples)oil_pressure        = np.random.normal(40, 8, n_samples)vibration            = np.random.normal(2.0, 0.9, n_samples)battery_voltage     = np.random.normal(12.6, 0.5, n_samples)coolant_temperature = np.random.normal(85, 10, n_samples)fuel_consumption    = np.random.normal(8, 1.5, n_samples)vehicle_speed        = np.random.normal(60, 20, n_samples)operating_hours      = np.random.uniform(0, 5000, n_samples)# Compute failure risk score and generate failure labelsrisk_score = (    0.09 * (engine_temperature - 90) +    0.07 * (coolant_temperature - 85) +    1.60 * (vibration - 2.0) +    -0.18 * (oil_pressure - 40) +    -2.20 * (battery_voltage - 12.6) +    0.0012 * operating_hours +    np.random.normal(0, 1.0, n_samples))prob_failure = 1 / (1 + np.exp(-(risk_score - 4.2)))failure = (np.random.rand(n_samples) < prob_failure).astype(int)df = pd.DataFrame({    'engine_temperature': engine_temperature,    'rpm': rpm,    'oil_pressure': oil_pressure,    'vibration': vibration,    'battery_voltage': battery_voltage,    'coolant_temperature': coolant_temperature,    'fuel_consumption': fuel_consumption,    'vehicle_speed': vehicle_speed,    'operating_hours': operating_hours,    'failure': failure})df.head()

In [ ]:
# Basic dataset statisticsdf.describe().T

In [ ]:
# Plot feature distributions by failure classfig, axes = plt.subplots(3, 3, figsize=(15, 11))features = df.columns[:-1]for ax, col in zip(axes.flatten(), features):    sns.histplot(data=df, x=col, hue='failure', bins=30, kde=True, ax=ax, palette=['#457b9d', '#e63946'])    ax.set_title(col)plt.tight_layout()plt.show()

In [ ]:
# Correlation heatmapplt.figure(figsize=(9, 7))sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)plt.title('Feature Correlation Heatmap')plt.tight_layout()plt.show()

In [ ]:
# Check missing values and split data into train/testprint(df.isnull().sum())X = df.drop(columns=['failure'])y = df['failure']X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train Logistic Regression modellog_reg = LogisticRegression(max_iter=2000)log_reg.fit(X_train_scaled, y_train)

In [ ]:
# Train Decision Tree modeldt = DecisionTreeClassifier(max_depth=7, random_state=42)dt.fit(X_train, y_train)

In [ ]:
# Train Random Forest modelrf = RandomForestClassifier(n_estimators=250, max_depth=9, random_state=42)rf.fit(X_train, y_train)

In [ ]:
# Train XGBoost modelxgb_model = xgb.XGBClassifier(    n_estimators=200, max_depth=4, learning_rate=0.1,    use_label_encoder=False, eval_metric='logloss', random_state=42)xgb_model.fit(X_train, y_train)

In [ ]:
# Build and train LSTM modelX_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))X_test_lstm  = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))lstm_model = Sequential([    LSTM(64, input_shape=(1, X_train.shape[1]), return_sequences=True),    Dropout(0.2),    LSTM(32),    Dropout(0.2),    Dense(16, activation='relu'),    Dense(1, activation='sigmoid')])lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])history = lstm_model.fit(    X_train_lstm, y_train,    validation_split=0.15,    epochs=25, batch_size=32, verbose=1)

In [ ]:
# Plot LSTM training accuracy and loss curvesplt.figure(figsize=(10,4))plt.subplot(1,2,1)plt.plot(history.history['accuracy'], label='train')plt.plot(history.history['val_accuracy'], label='val')plt.title('LSTM Accuracy'); plt.xlabel('Epoch'); plt.legend()plt.subplot(1,2,2)plt.plot(history.history['loss'], label='train')plt.plot(history.history['val_loss'], label='val')plt.title('LSTM Loss'); plt.xlabel('Epoch'); plt.legend()plt.tight_layout(); plt.show()

In [ ]:
# Evaluate and compare all modelsdef evaluate(name, y_true, y_pred, y_proba):    return {        'Model': name,        'Accuracy': accuracy_score(y_true, y_pred),        'Precision': precision_score(y_true, y_pred),        'Recall': recall_score(y_true, y_pred),        'F1-Score': f1_score(y_true, y_pred),    }results = []pred = log_reg.predict(X_test_scaled); proba = log_reg.predict_proba(X_test_scaled)[:,1]results.append(evaluate('Logistic Regression', y_test, pred, proba))pred = dt.predict(X_test); proba = dt.predict_proba(X_test)[:,1]results.append(evaluate('Decision Tree', y_test, pred, proba))pred = rf.predict(X_test); proba = rf.predict_proba(X_test)[:,1]results.append(evaluate('Random Forest', y_test, pred, proba))pred = xgb_model.predict(X_test); proba = xgb_model.predict_proba(X_test)[:,1]results.append(evaluate('XGBoost', y_test, pred, proba))lstm_proba = lstm_model.predict(X_test_lstm).flatten()lstm_pred = (lstm_proba >= 0.5).astype(int)results.append(evaluate('LSTM', y_test, lstm_pred, lstm_proba))results_df = pd.DataFrame(results).set_index('Model').round(3)results_df

In [ ]:
# Plot accuracy comparison across modelsresults_df['Accuracy'].plot(kind='bar', figsize=(7,4.5), color=['#8d99ae','#457b9d','#1d3557','#e63946','#2a9d8f'])plt.ylabel('Accuracy'); plt.title('Model Accuracy Comparison'); plt.ylim(0,1)plt.xticks(rotation=20, ha='right')plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix for best modelbest_model_name = results_df['Accuracy'].idxmax()print("Best performing model:", best_model_name)cm = confusion_matrix(y_test, rf.predict(X_test))plt.figure(figsize=(4.5,4))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Healthy','Failure'], yticklabels=['Healthy','Failure'])plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix - Random Forest')plt.tight_layout(); plt.show()print(classification_report(y_test, rf.predict(X_test), target_names=['Healthy','Failure']))

In [ ]:
# ROC curve comparison across all modelsplt.figure(figsize=(6,5))models_probas = {    'Logistic Regression': log_reg.predict_proba(X_test_scaled)[:,1],    'Decision Tree': dt.predict_proba(X_test)[:,1],    'Random Forest': rf.predict_proba(X_test)[:,1],    'XGBoost': xgb_model.predict_proba(X_test)[:,1],    'LSTM': lstm_proba}for name, proba in models_probas.items():    fpr, tpr, _ = roc_curve(y_test, proba)    plt.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.2f})')plt.plot([0,1],[0,1],'k--', linewidth=0.8)plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')plt.title('ROC Curve Comparison'); plt.legend(fontsize=8)plt.tight_layout(); plt.show()

In [ ]:
# Explainable AI - SHAP feature importanceimport shapexplainer = shap.TreeExplainer(rf)shap_values = explainer.shap_values(X_test)shap.summary_plot(shap_values[1] if isinstance(shap_values, list) else shap_values,                   X_test, plot_type='bar', show=True)

In [ ]:
# Vehicle health score and maintenance alert generatordef vehicle_health_report(sample_row: pd.Series, model=rf, feature_names=X.columns):    x = sample_row[feature_names].values.reshape(1, -1)    fail_prob = model.predict_proba(x)[0][1]    health_score = round((1 - fail_prob) * 100, 1)    if health_score >= 80:        status = "NORMAL"    elif health_score >= 50:        status = "WARNING"    else:        status = "CRITICAL"    healthy_mean = X_train[y_train == 0].mean()    deviation = ((sample_row[feature_names] - healthy_mean) / healthy_mean).abs()    importance = pd.Series(model.feature_importances_, index=feature_names)    contribution = (deviation * importance).sort_values(ascending=False)    print(f"Vehicle Health: {health_score}%")    print(f"Failure Probability: {round(fail_prob*100,1)}%")    print(f"Status: {status}")    print("Major contributing factors:")    for i, (feat, _) in enumerate(contribution.head(3).items(), 1):        print(f"  {i}. {feat.replace('_',' ').title()}")    return health_score, fail_prob, statussample = X_test.iloc[np.argmax(rf.predict_proba(X_test)[:,1])]_ = vehicle_health_report(sample)

In [ ]:
# Save trained models for deploymentimport joblibjoblib.dump(rf, 'vehicle_health_rf_model.pkl')joblib.dump(scaler, 'feature_scaler.pkl')lstm_model.save('vehicle_health_lstm_model.h5')